# 02 · multilingual-e5-small embeddings for a diverse S1 / S2 / S3 sample

**Input:** the per-country parquet files written by `01_eda_preprocessing.ipynb` into `processed/`
(`train_s1_US.parquet`, `test_s1_France.parquet`, `train_s2_India.parquet`, …), plus
`train_ground_truth.tsv` from the competition data (used only to link the sampled records).

**Output** (`embeddings/multilingual-e5-small/`):

| File | Content |
|---|---|
| `s1_embeddings.npy`, `s2_embeddings.npy`, `s3_embeddings.npy` | float16, one unit-norm row per sampled record (384 numbers for e5-small, 768 for e5-base, 1024 for e5-large) |
| `s1_meta.parquet`, `s2_meta.parquet`, `s3_meta.parquet` | row-aligned metadata: `entity_id`, split, country, `kind`, true S1 link, diversity tags, embedded text |
| `manifest.json` | settings, counts, tag shares (sample vs population), timing |
| `faiss/s1_buckets.parquet` | per S1: its top-`TOP_K` S2/S3 candidates (the "bucket") with rank, retrieval score and original e5 cosine (`BUCKET_METHOD`, IVF-PQ by default) |
| `faiss/s1_buckets_exact.parquet`, `faiss/s1_buckets_ivfpq.parquet` | the buckets from each search method (the IVF-PQ file only when `BUCKET_METHOD=ivfpq`) |
| `faiss/mean_<country>.npy`, `faiss/index_<country>.faiss` | the mean used to center that country's embeddings, and the IVF-PQ index over its centered S2+S3 embeddings (only when `BUCKET_METHOD=ivfpq`) |
| `faiss/bucket_recall.csv` | share of true matches inside the buckets, IVF-PQ vs exact search |
| `faiss/s1_buckets_all.parquet` | (with `EXTRA_SEARCHES`) the buckets the full pipeline judges: e5 top-30 plus the reverse / address / name / number / empty-address searches, in 02 full's format (`rank` by score, `score`, `cosine`, and which searches found each pair: `e5_rank`, `rev_rank`, `addr_key`, `name_rank`, `num_key`, `empty_rank`, `empty_rev_rank`). `03_lightgbm_matcher.ipynb` trains on it with `BUCKETS=all` (or `sub2`) |
| `faiss/extra_recall.csv` | e5 top-k vs e5 top-30 + each extra search: pairs per S1, share of true matches inside, ceiling |
| `faiss/metric_f05.csv` | the challenge metric (macro F0.5 per S1) for the buckets: ceiling, baselines, tuned cosine cutoff |

**Sample size: the "Sample size" block at the top of the first code cell** is the only place to change it:

| Setting | Default | Meaning |
|---|---|---|
| `SAMPLE_SIZE` | `100k` | records per source (S1, S2, S3): a number (`250000`), `k` / `m` (`20k`, `500k`, `1.5m`) or a preset: `tiny` = 10k, `small` = 100k, `medium` = 500k, `large` = 1m |
| `N_S1`, `N_S2`, `N_S3` | empty | optional size of one source, same format (e.g. more candidates than S1) |
| `COUNTRIES` | all | optional, e.g. `India,US`: sample only these countries (each source's quota is split over them) |
| `SAMPLE_NAME` | from the sizes | name of the output folder, e.g. `sample_100k`, `sample_s1-50k_s2-200k_s3-200k_India` |

Every sample is saved in its own folder (`embeddings/multilingual-e5-small/<SAMPLE_NAME>/`), so runs with
different sizes do not overwrite each other, and `03_lightgbm_matcher.ipynb` can pick one by name. The
"Sample plan" table (after the inventory) shows, per source and country, the requested and available
records before anything is read.

**How the sample is made diverse** (`SAMPLE_SIZE` records per source):
* **Countries:** each source's quota is split evenly across the countries it has. France exists
  only in test S1, so it appears in S1 only, unless `test_s2_*` / `test_s3_*` files are added
  to `processed/` (they are picked up automatically).
* **Linked groups:** S1 records of labeled countries come from train. For S2/S3, 70% are records
  whose true S1 is in the S1 sample (so true pairs exist inside the sample), 30% are no-match
  records. Any shortfall is filled with records owned by an S1 outside the sample.
* **Hard cases:** within every (source, country) quota, each diversity tag gets at least
  `MIN_TAG_SHARE` (5%) when available: native-script name, web/handle name, empty address,
  honorific, landmark address, postcode present, no legal form, one-word name. The rest is random.

**Text given to e5** (`TEXT_MODE`):
* `fixed` (default, the same text as `02_full_e5_buckets.ipynb`): `name_norm | addr_norm` from
  `processed/`, except that names and addresses written in an Indian script are transliterated to
  Latin letters from the original text (`anyascii` with two fixes: Malayalam "tt", nasal "n")
* `processed`: `name_norm | addr_norm` from `processed/` as they are
* `latin`: `name_lat legal_form | addr_norm` (ASCII transliteration)
* `raw`: `business_name | business_address`

The `processed` step strips combining marks, which also removes the vowel signs of Indian scripts,
so with `processed` their names reach e5 garbled. Use `fixed` to compare models or settings with
the pipeline that makes the submission.

**Run on Kaggle:**
1. Attach the competition dataset and the `processed/` output of notebook 01. Add the 01
   notebook's output as an input, or run 01 in this session.
2. Accelerator: GPU T4 x2; both GPUs are used automatically for encoding and search. Internet on,
   to download the models (MIT); or attach a model and give its folder in `E5_MODELS`.

**Which e5 models** (`E5_MODELS`, default `small,base,large`): the sample is built once, then the
embedding, quick check, bucket and metric sections run **once per model, in order, in the same run**
("Run every model"). `small` has 118M parameters and 384 numbers per text, `base` 278M / 768, `large`
560M / 1024; any Hugging Face id or local folder also works. Bigger models are slower to encode (base
about 3x, large about 8x small). Every model writes to its own folder (`embeddings/<model>/<sample>/`),
so all of them are in this run's output, and the last section compares them side by side. A model
whose embeddings are already saved there (same records, texts and model) is loaded, not re-encoded.
`E5_MODELS = "small"` runs one model.
3. Run all. Encoding 300K short texts takes a few minutes.

**Bigger sample** (e.g. 500K per source): set `SAMPLE_SIZE = "500k"`. `BUCKET_METHOD = "auto"` then
uses exact search only and skips the IVF-PQ comparison, which is the slow CPU part above 200K per
source. Samples above 40K per country need 01's `FULL_RUN` output, since the sample-mode files hold
only 40K rows each (the sample plan warns about this).

Settings live in the first code cell and can also be set as environment variables: `PROCESSED_DIR`,
`DATA_ROOT`, `EMB_DIR`, `SAMPLE_SIZE`, `N_S1`, `N_S2`, `N_S3`, `COUNTRIES`, `SAMPLE_NAME`, `TEXT_MODE`,
`E5_MODELS`, `TOP_K`, `NPROBE_FRAC`, `BUCKET_METHOD`.

The last part retrieves, for every sampled S1, its top-k S2/S3 records per country as that S1's
candidate bucket: exact search (on all GPUs when there are any), and a **FAISS IVF-PQ** index for
comparison. `faiss-cpu` (MIT) is installed if missing.

In [ ]:
import csv
import glob
import json
import os
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch

# ---- Sample size: change it here --------------------------------------------------------------------
SAMPLE_SIZE = os.environ.get("SAMPLE_SIZE", "100k")      # records per source: 250000 | 20k | 500k | 1.5m | tiny/small/medium/large
N_S1 = os.environ.get("N_S1", "")                        # optional per-source sizes (same format; empty = SAMPLE_SIZE)
N_S2 = os.environ.get("N_S2", "")
N_S3 = os.environ.get("N_S3", "")
COUNTRIES = [c.strip() for c in os.environ.get("COUNTRIES", "").split(",") if c.strip()]   # empty = every country
SAMPLE_NAME = os.environ.get("SAMPLE_NAME", "")          # output folder name; empty = built from the sizes
# -------------------------------------------------------------------------------------------------------
PRESETS = {"tiny": 10_000, "small": 100_000, "medium": 500_000, "large": 1_000_000}


def parse_size(value):
    """'100k' -> 100000, '1.5m' -> 1500000, '250000' -> 250000, 'small' -> 100000."""
    v = str(value).strip().lower().replace("_", "").replace(",", "")
    if v in PRESETS:
        return PRESETS[v]
    scale = {"k": 1_000, "m": 1_000_000}.get(v[-1:], 1)
    n = int(float(v[:-1] if scale > 1 else v) * scale)
    assert n > 0, f"sample size must be positive: {value!r}"
    return n


def size_label(n):
    """100000 -> '100k', 1500000 -> '1.5m', 12345 -> '12345'."""
    for unit, div in (("m", 1_000_000), ("k", 1_000)):
        if n >= div and n % (div // 10) == 0:
            return f"{n / div:g}{unit}"
    return str(n)


N_BY_SOURCE = {s: parse_size(v or SAMPLE_SIZE) for s, v in (("s1", N_S1), ("s2", N_S2), ("s3", N_S3))}
if not SAMPLE_NAME:
    sizes = set(N_BY_SOURCE.values())
    SAMPLE_NAME = "sample_" + (size_label(sizes.pop()) if len(sizes) == 1
                               else "_".join(f"{s}-{size_label(n)}" for s, n in N_BY_SOURCE.items()))
    SAMPLE_NAME += "".join(f"_{c}" for c in COUNTRIES)
SEED = 42
LINKED_SHARE = 0.70          # S2/S3: share whose true S1 is in the S1 sample
MIN_TAG_SHARE = 0.05         # minimum share per diversity tag within each (source, country) quota
TEXT_MODE = os.environ.get("TEXT_MODE", "fixed")               # fixed (as 02 full) | processed | latin | raw
E5_MODEL_IDS = {"small": "intfloat/multilingual-e5-small", "base": "intfloat/multilingual-e5-base",
             "large": "intfloat/multilingual-e5-large"}                  # all MIT; 118M / 278M / 560M parameters
MODEL_LIST = [m.strip() for m in os.environ.get("E5_MODELS", os.environ.get("E5_MODEL", "small,base,large")).split(",")
              if m.strip()]                             # models run one after the other: small, base, large or model ids
MODEL_NAME = E5_MODEL_IDS.get(MODEL_LIST[0], MODEL_LIST[0])
PREFIX = "query: "           # E5 expects "query: " for symmetric similarity
MAX_SEQ_LENGTH = 128         # tokens; names + addresses are short
TOP_K = int(os.environ.get("TOP_K", 50))                      # bucket size per S1
WORD_MAP = os.environ.get("WORD_MAP", "true") == "true"      # fixed text: map transliterated Indian-script words to English
EXTRA_SEARCHES = os.environ.get("EXTRA_SEARCHES", "true") == "true"   # also run 02 full's reverse / address / name searches
BUCKET_METHOD = os.environ.get("BUCKET_METHOD", "auto")        # auto | ivfpq (IVF-PQ + exact, compared) | exact (exact only)
if BUCKET_METHOD == "auto":                                    # the IVF-PQ comparison is slow on the CPU for big samples
    BUCKET_METHOD = "ivfpq" if max(N_BY_SOURCE.values()) <= 200_000 else "exact"
NPROBE_FRAC = float(os.environ.get("NPROBE_FRAC", 0.25))       # share of IVF clusters scanned per query
assert TEXT_MODE in {"fixed", "processed", "latin", "raw"}, TEXT_MODE
assert BUCKET_METHOD in {"ivfpq", "exact"}, BUCKET_METHOD


def find_processed_dir():
    """Folder with 01's per-country parquet: $PROCESSED_DIR, ./processed, /kaggle/working, any Kaggle input."""
    candidates = [Path(os.environ["PROCESSED_DIR"])] if os.environ.get("PROCESSED_DIR") else []
    candidates += [Path.cwd() / "processed", Path("/kaggle/working/processed")]
    candidates += sorted({Path(p).parent for p in glob.glob("/kaggle/input/**/train_s1_*.parquet", recursive=True)})
    for c in candidates:
        if c.is_dir() and any(c.glob("train_s1_*.parquet")):
            return c
    raise FileNotFoundError("processed/ not found: run 01_eda_preprocessing.ipynb or set PROCESSED_DIR")


def find_ground_truth():
    """train_ground_truth.tsv under $DATA_ROOT, /kaggle/input, or the local dataset folders used by 01."""
    roots = [os.environ.get("DATA_ROOT", ""), "/kaggle/input"]
    here = Path.cwd().resolve()
    roots += [str(here.parent / "6ab10eb3b23ba_student_resource" / "student_resource" / "dataset"),
              str(here.parent / "Data" / "student_resource" / "dataset")]
    for root in roots:
        if root and os.path.isdir(root):
            hits = [h for h in glob.glob(os.path.join(root, "**", "train_ground_truth.tsv"), recursive=True)
                    if "__MACOSX" not in h]
            if hits:
                return Path(sorted(hits, key=len)[0])
    raise FileNotFoundError("train_ground_truth.tsv not found; set DATA_ROOT")


PROCESSED_DIR = find_processed_dir()
GT_FILE = find_ground_truth()
EMB_ROOT = Path(os.environ.get("EMB_DIR") or ("/kaggle/working/embeddings" if os.path.isdir("/kaggle/working")
                                              else "embeddings"))


def set_model(name):
    """Switch to one e5 model: MODEL_NAME and its output folder embeddings/<model>/<sample>/."""
    global MODEL_NAME, OUT_DIR
    MODEL_NAME = E5_MODEL_IDS.get(name, name)
    OUT_DIR = EMB_ROOT / MODEL_NAME.rstrip("/").split("/")[-1] / SAMPLE_NAME
    OUT_DIR.mkdir(parents=True, exist_ok=True)


set_model(MODEL_LIST[0])
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPUS = [f"cuda:{i}" for i in range(torch.cuda.device_count())] if DEVICE == "cuda" else []   # all used (T4 x2 on Kaggle)
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu"
# fp16 speeds up tensor-core GPUs (T4, A100) but is ~4x slower on GTX 16xx cards (measured on a GTX 1650).
USE_FP16 = DEVICE == "cuda" and "GTX 16" not in GPU_NAME
BATCH_SIZE = 1024 if USE_FP16 else 512                         # texts per GPU batch when encoding
print(f"processed: {PROCESSED_DIR}\nground truth: {GT_FILE}\noutput: {EMB_ROOT}/<model>/{SAMPLE_NAME}\n"
      f"models, in order: {[E5_MODEL_IDS.get(m, m) for m in MODEL_LIST]}")
print(f"device: {len(GPUS)} x {GPU_NAME} | fp16: {USE_FP16} | batch: {BATCH_SIZE} | text mode: {TEXT_MODE}")
print(f"sample: {SAMPLE_NAME} | " + " | ".join(f"{s.upper()} {n:,}" for s, n in N_BY_SOURCE.items())
      + f" | countries: {', '.join(COUNTRIES) or 'all'} | buckets: {BUCKET_METHOD}, top {TOP_K}")

## Challenge metric: macro F0.5 per S1
From the problem statement: for every S1, F0.5 = 1.25·P·R / (0.25·P + R) over its predicted vs
true S2/S3 IDs, then averaged over **all** S1 (macro). An S1 with no true match scores 1.0 for an
empty prediction and 0.0 for any prediction. An S1 with matches scores 0.0 for an empty prediction.
The assert reproduces the worked example in the problem statement (0.714).

In [ ]:
def f05(predicted, true):
    """Challenge F0.5 for one S1: predicted and true are collections of S2/S3 entity IDs."""
    predicted, true = set(predicted), set(true)
    if not true:
        return 1.0 if not predicted else 0.0
    tp = len(predicted & true)
    if tp == 0:
        return 0.0
    precision, recall = tp / len(predicted), tp / len(true)
    return 1.25 * precision * recall / (0.25 * precision + recall)


def macro_f05(pred_by_s1, true_by_s1, s1_ids):
    """Mean challenge F0.5 over s1_ids (S1 missing from pred_by_s1 = empty prediction)."""
    return float(np.mean([f05(pred_by_s1.get(s, ()), true_by_s1.get(s, ())) for s in s1_ids]))


assert round(f05(["S2-00047", "S2-00193", "S3-00812"], ["S2-00047", "S3-00812"]), 3) == 0.714
assert f05([], []) == 1.0 and f05(["S2-1"], []) == 0.0 and f05([], ["S2-1"]) == 0.0
print("challenge metric OK (problem-statement example = 0.714)")

## Inventory of `processed/`

In [ ]:
FILE_RE = re.compile(r"^(train|test)_(s[123])_(.+)\.parquet$")
inventory = []
for f in sorted(PROCESSED_DIR.glob("*.parquet")):
    m = FILE_RE.match(f.name)
    if m:
        inventory.append({"split": m.group(1), "source": m.group(2), "country": m.group(3), "path": f,
                          "rows": pq.ParquetFile(f).metadata.num_rows})
INVENTORY = pd.DataFrame(inventory)
assert len(INVENTORY), f"no <split>_<source>_<country>.parquet files in {PROCESSED_DIR}"
if COUNTRIES:
    missing = set(COUNTRIES) - set(INVENTORY["country"])
    assert not missing, f"COUNTRIES not in processed/: {sorted(missing)} (available: {sorted(set(INVENTORY['country']))})"
    INVENTORY = INVENTORY[INVENTORY["country"].isin(COUNTRIES)].reset_index(drop=True)
if (INVENTORY["rows"] <= 40_000).all():
    print("NOTE: every file has <= 40,000 rows, so processed/ looks like 01's sample mode (first 40K rows "
          "per country). Few true S1-S2/S3 pairs will exist. Run 01 with FULL_RUN=true for linked groups.")
display(INVENTORY.drop(columns="path"))

# sample plan: each source's size split evenly over its countries, vs the records available
plan = []
for source, n in N_BY_SOURCE.items():
    countries = sorted(INVENTORY.loc[INVENTORY["source"] == source, "country"].unique())
    base, extra = divmod(n, max(len(countries), 1))
    for i, country in enumerate(countries):
        quota = base + (i < extra)
        available = int(INVENTORY.loc[(INVENTORY["source"] == source) & (INVENTORY["country"] == country), "rows"].sum())
        plan.append({"source": source, "country": country, "requested": quota, "available": available,
                     "sampled": min(quota, available)})
PLAN = pd.DataFrame(plan)
short = PLAN[PLAN["requested"] > PLAN["available"]]
if len(short):
    print(f"NOTE: {len(short)} source/country quotas ask for more records than exist; those take everything "
          "available (see 'sampled'). Lower SAMPLE_SIZE or use 01's FULL_RUN output.")
PLAN

## Loading and diversity tags

In [ ]:
READ_COLS = ["entity_id", "business_name", "business_address", "country", "name_norm", "core_name",
             "legal_form", "domain_stem", "name_lat", "has_honorific", "addr_norm", "empty_addr",
             "missing_pin", "landmark_flag"]
STR_COLS = ["entity_id", "business_name", "business_address", "country", "name_norm", "core_name",
            "legal_form", "domain_stem", "name_lat", "addr_norm"]
INT_COLS = ["has_honorific", "empty_addr", "missing_pin", "landmark_flag"]
# Anything outside Latin, Latin-1/Extended and general punctuation counts as a native script.
RE_NATIVE = re.compile(r"[^\x00-\x7f\u00a0-\u024f\u1e00-\u1eff\u2000-\u206f\u20a0-\u20cf]")
RE_HANDLE = re.compile(r"^\s*[@#]\S+\s*$")
TAGS = ["tag_native_script", "tag_web_name", "tag_empty_address", "tag_honorific", "tag_landmark",
        "tag_has_postcode", "tag_no_legal_form", "tag_one_word_name"]


def read_processed(path):
    """Read one processed parquet with the needed columns; missing strings become '' and flags 0."""
    available = set(pq.read_schema(path).names)
    df = pd.read_parquet(path, columns=[c for c in READ_COLS if c in available])
    for c in STR_COLS:
        df[c] = df[c].fillna("").astype(str) if c in df else ""
    for c in INT_COLS:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int) if c in df else 0
    return df


def add_tags(df):
    """Boolean diversity tags computed from the processed columns."""
    names = df["business_name"].tolist()
    return df.assign(
        tag_native_script=[RE_NATIVE.search(x) is not None for x in names],
        tag_web_name=(df["domain_stem"] != "").to_numpy() | np.array([RE_HANDLE.match(x) is not None for x in names]),
        tag_empty_address=(df["empty_addr"] == 1) | (df["addr_norm"] == ""),
        tag_honorific=df["has_honorific"] == 1,
        tag_landmark=df["landmark_flag"] == 1,
        tag_has_postcode=df["missing_pin"] == 0,
        tag_no_legal_form=df["legal_form"] == "",
        tag_one_word_name=[len(x.split()) <= 1 for x in df["core_name"].tolist()])


def diverse_sample(df, n, rng):
    """Row labels of n rows: at least MIN_TAG_SHARE * n rows of every tag (when available), rest random."""
    if n <= 0 or len(df) == 0:
        return df.index[:0]
    if len(df) <= n:
        return df.index
    chosen = np.zeros(len(df), bool)
    per_tag = int(MIN_TAG_SHARE * n)
    for tag in TAGS:
        pool = np.flatnonzero(df[tag].to_numpy() & ~chosen)
        take = min(per_tag, len(pool))
        if take:
            chosen[rng.choice(pool, take, replace=False)] = True
    rest = np.flatnonzero(~chosen)
    need = n - int(chosen.sum())
    if need > 0:
        chosen[rng.choice(rest, min(need, len(rest)), replace=False)] = True
    return df.index[chosen]


def split_quota(total, keys):
    """Split an integer total as evenly as possible over keys."""
    base, extra = divmod(total, len(keys))
    return {k: base + (i < extra) for i, k in enumerate(keys)}


def tag_shares(df):
    """Percentage of rows carrying each tag."""
    return {t: round(100 * float(df[t].mean()), 2) for t in TAGS} if len(df) else {}

## Ground truth (links only)
Used to link sampled S2/S3 records to their S1 and to find no-match records. It is not used to build the text.

In [ ]:
gt = pd.read_csv(GT_FILE, sep="\t", dtype=object, keep_default_na=False, na_filter=False,
                 quoting=csv.QUOTE_NONE)
n_matches_of = gt.set_index("source1_entity_id")["matched_entity_ids"].map(
    lambda s: len(s.split(",")) if s else 0)
gt = gt[gt["matched_entity_ids"] != ""]
pairs = gt.assign(entity_id=gt["matched_entity_ids"].str.split(",")).explode("entity_id")
OWNER = pd.Series(pairs["source1_entity_id"].to_numpy(), index=pairs["entity_id"].to_numpy())
del gt, pairs
print(f"{len(OWNER):,} S2/S3 records have a true S1; {int((n_matches_of == 0).sum()):,} train S1 are singletons")

## Sample S1
Labeled countries are drawn from train (so their S2/S3 matches can be linked). Countries that
exist only in test (France) are drawn from test.

In [ ]:
rng = np.random.default_rng(SEED)
POPULATION_TAGS = {}


def files_for(source, country):
    """Processed files of one source and country, train first."""
    sel = INVENTORY[(INVENTORY["source"] == source) & (INVENTORY["country"] == country)]
    return sel.sort_values("split", key=lambda s: s.map({"train": 0, "test": 1}))


s1_parts = []
s1_countries = sorted(INVENTORY.loc[INVENTORY["source"] == "s1", "country"].unique())
for country, quota in split_quota(N_BY_SOURCE["s1"], s1_countries).items():
    for _, f in files_for("s1", country).iterrows():
        if quota <= 0:
            break
        df = add_tags(read_processed(f["path"]))
        POPULATION_TAGS[f"s1/{f['split']}/{country}"] = tag_shares(df)
        picked = df.loc[diverse_sample(df, quota, rng)].assign(split=f["split"])
        s1_parts.append(picked)
        quota -= len(picked)
        del df
S1 = pd.concat(s1_parts, ignore_index=True)
S1["n_true_matches"] = S1["entity_id"].map(n_matches_of).where(S1["split"] == "train")
S1["kind"] = np.where(S1["split"] != "train", "unlabeled",
                      np.where(S1["n_true_matches"] > 0, "has_matches", "singleton"))
SAMPLED_S1 = set(S1["entity_id"])
S1.groupby(["country", "split", "kind"]).size().rename("records").to_frame()

## Sample S2 and S3
Per country: 70% records linked to a sampled S1, 30% no-match records, each chosen with the tag
minimums. Shortfalls are filled with records whose S1 is outside the sample, then with anything
left. Countries that exist only in test contribute unlabeled records.

In [ ]:
def sample_source(source):
    """Diverse, partly linked sample of one of S2/S3."""
    parts = []
    countries = sorted(INVENTORY.loc[INVENTORY["source"] == source, "country"].unique())
    for country, quota in split_quota(N_BY_SOURCE[source], countries).items():
        frames = []
        for _, f in files_for(source, country).iterrows():
            df = add_tags(read_processed(f["path"])).assign(split=f["split"])
            POPULATION_TAGS[f"{source}/{f['split']}/{country}"] = tag_shares(df)
            frames.append(df)
        pool = pd.concat(frames, ignore_index=True)
        del frames
        owner = pool["entity_id"].map(OWNER).where(pool["split"] == "train")
        pool["true_s1"] = owner.fillna("")
        pool["kind"] = np.select(
            [pool["split"] != "train", owner.isna(), owner.isin(SAMPLED_S1)],
            ["unlabeled", "no_match", "linked"], default="other_owner")
        plan = [("linked", round(LINKED_SHARE * quota)), ("no_match", quota - round(LINKED_SHARE * quota))]
        picked = []
        for kind, n in plan:
            idx = diverse_sample(pool[pool["kind"] == kind], n, rng)
            picked.append(idx)
        taken = pool.index.isin(np.concatenate([np.asarray(i) for i in picked]))
        for kind in ("other_owner", "linked", "no_match", "unlabeled"):   # fill any shortfall
            short = quota - int(taken.sum())
            if short <= 0:
                break
            idx = diverse_sample(pool[(pool["kind"] == kind) & ~taken], short, rng)
            taken |= pool.index.isin(idx)
        parts.append(pool[taken])
        n_linked = int((pool[taken]["kind"] == "linked").sum())
        print(f"{source} {country}: {int(taken.sum()):,} records ({n_linked:,} linked to a sampled S1)")
        del pool
    return pd.concat(parts, ignore_index=True)


S2 = sample_source("s2")
S3 = sample_source("s3")
del OWNER, n_matches_of     # only needed to build the sample; not read again in embed/bucket/recall/metric/extra
pd.concat({"s2": S2.groupby(["country", "kind"]).size(), "s3": S3.groupby(["country", "kind"]).size()},
          axis=1).fillna(0).astype(int)

### Diversity check: tag shares in the sample vs the full files

In [ ]:
rows = []
for source, df in (("s1", S1), ("s2", S2), ("s3", S3)):
    for country, g in df.groupby("country"):
        rows.append({"source": source, "country": country, "records": len(g), **tag_shares(g)})
SAMPLE_TAGS = pd.DataFrame(rows).set_index(["source", "country"])
print("sample (% of records with each tag):")
display(SAMPLE_TAGS)
print("full processed files (% of records with each tag):")
pd.DataFrame(POPULATION_TAGS).T

## Embed with the e5 model
The sections from here to the metric define one step each (`embed_step`, `check_step`, `bucket_step`,
`recall_step`, `metric_step`); "Run every model" below runs them for each model in `E5_MODELS`.

GPU use:
* Each unique text is encoded once, in fp16 on tensor-core GPUs (T4), `BATCH_SIZE` texts per batch.
* Texts are sorted by length first, so each batch holds texts of similar length and little padding.
* With several GPUs (Kaggle T4 x2), one encoding process runs per GPU. Chunks of `CHUNK_SIZE`
  texts are handed out to whichever GPU is free, and the processes stop once all sources are embedded.

In [ ]:
def embed_step():
    global CHUNK_SIZE, EMB, META_COLS, REUSE_EMBEDDINGS, RE_ANUSVARA_N, RE_NATIVE, SentenceTransformer, TIMING
    global _ANUSVARA, _INDIC_BLOCKS, _LABIALS, _LATIN_RANGES, _model, _pool, a, anyascii, b, build_text, c
    global cached, d, df, embed, extra, get_model, get_pool, importlib, inspect, k, manifest, o, r, s
    global saved_embeddings, secs, source, stop_pool, subprocess, sys, translit, v
    import inspect

    from sentence_transformers import SentenceTransformer

    CHUNK_SIZE = 16 * BATCH_SIZE     # texts per work unit handed to one GPU process


    import importlib.util  # noqa: E402
    import subprocess  # noqa: E402
    import sys  # noqa: E402
    if importlib.util.find_spec("anyascii") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "anyascii"])
    from anyascii import anyascii  # noqa: E402  (ISC)
    # a text has a native script when it has a character outside Latin, Latin-1/Extended and general punctuation
    _LATIN_RANGES = [(0x00, 0x7F), (0xA0, 0x24F), (0x1E00, 0x1EFF), (0x2000, 0x206F), (0x20A0, 0x20CF)]
    RE_NATIVE = re.compile("[^" + "".join(re.escape(chr(a)) + "-" + re.escape(chr(b)) for a, b in _LATIN_RANGES) + "]")
    # Two fixes to anyascii for Indian scripts (measured on true pairs, Latin S1 name vs Indian-script candidate name,
    # mean token-set similarity: Malayalam 66.1 -> 72.5, Telugu 75.5 -> 76.7):
    # * Malayalam റ്റ is read "tt"; anyascii gives "rr" ("ലിമിറ്റഡ്" -> "limirrd" instead of "limittd").
    # * An anusvara (the nasal dot) before a consonant other than p/ph/b/bh/m is read "n"; anyascii always gives "m"
    #   ("आनंद" -> "anmd" instead of "annd", "सेंटर" -> "semtr" instead of "sentr").
    _INDIC_BLOCKS = range(0x0900, 0x0D80, 0x80)                     # Devanagari, Bengali, ..., Malayalam (same layout)
    _ANUSVARA = "".join(chr(b + 0x02) for b in _INDIC_BLOCKS)
    _LABIALS = "".join(chr(b + o) for b in _INDIC_BLOCKS for o in range(0x2A, 0x2F))      # pa, pha, ba, bha, ma
    RE_ANUSVARA_N = re.compile(f"[{_ANUSVARA}](?=[\u0900-\u0D7F])(?![{_LABIALS}])")


    def translit(s):
        """anyascii with the two fixes above."""
        return anyascii(RE_ANUSVARA_N.sub("n", s.replace("\u0D31\u0D4D\u0D31", "\u0D1F\u0D4D\u0D1F")))


    def build_text(df):
        """Text given to e5, according to TEXT_MODE."""
        if TEXT_MODE == "fixed":                    # as 02 (full): Indian scripts transliterated from the original text
            lat = lambda raw, clean: " ".join(translit(raw).lower().split()) if RE_NATIVE.search(raw) else clean  # noqa: E731
            ensure_word_map()
            names = [translit_name(r) if RE_NATIVE.search(r) else c
                     for r, c in zip(df["business_name"].fillna(""), df["name_norm"].fillna(""))]
            addrs = [lat(r, c) for r, c in zip(df["business_address"].fillna(""), df["addr_norm"].fillna(""))]
            return [f"{n.strip()} | {a.strip()}" if a.strip() else n.strip() for n, a in zip(names, addrs)]
        if TEXT_MODE == "raw":
            names, addrs = df["business_name"], df["business_address"]
        elif TEXT_MODE == "latin":
            names, addrs = (df["name_lat"] + " " + df["legal_form"]).str.strip(), df["addr_norm"]
        else:
            names, addrs = df["name_norm"], df["addr_norm"]
        return [f"{n.strip()} | {a.strip()}" if a.strip() else n.strip() for n, a in zip(names, addrs)]


    REUSE_EMBEDDINGS = True      # load saved embeddings when the same records, texts and model were embedded before
    _model, _pool = None, None


    def get_model():
        """Load the e5 model once, on first use (kept on the CPU when it is copied to several GPUs)."""
        global _model
        if _model is None:
            _model = SentenceTransformer(MODEL_NAME, device="cpu" if len(GPUS) > 1 else DEVICE)
            _model.max_seq_length = MAX_SEQ_LENGTH
            if USE_FP16:
                _model.half()
        return _model


    def get_pool():
        """One encoding process per GPU, started once and reused for S1, S2 and S3."""
        global _pool
        if _pool is None:
            _pool = get_model().start_multi_process_pool(target_devices=GPUS)
        return _pool


    def stop_pool():
        """Stop the encoding processes (frees their GPU memory for the search below)."""
        global _pool
        if _pool is not None:
            SentenceTransformer.stop_multi_process_pool(_pool)
            _pool = None


    def embed(texts):
        """Unit-norm float16 embeddings aligned with texts, and seconds taken.

        Each unique text is encoded once, longest first, so batches hold texts of similar length.
        With several GPUs the chunks are shared among one process per GPU.
        """
        uniq, inverse = np.unique(np.asarray(texts, dtype=object), return_inverse=True)
        order = np.argsort([-len(t) for t in uniq], kind="stable")
        batch = [PREFIX + t for t in uniq[order]]
        t0 = time.time()
        model = get_model()
        if len(GPUS) > 1:
            if "pool" in inspect.signature(model.encode).parameters:          # sentence-transformers >= 5
                vecs = model.encode(batch, pool=get_pool(), batch_size=BATCH_SIZE, chunk_size=CHUNK_SIZE,
                                    convert_to_numpy=True, show_progress_bar=True)
            else:
                vecs = model.encode_multi_process(batch, get_pool(), batch_size=BATCH_SIZE, chunk_size=CHUNK_SIZE)
        else:
            vecs = model.encode(batch, batch_size=BATCH_SIZE, convert_to_numpy=True, show_progress_bar=True)
        vecs = np.asarray(vecs, dtype=np.float32)
        vecs /= np.maximum(np.linalg.norm(vecs, axis=1, keepdims=True), 1e-12)
        in_uniq_order = np.empty_like(vecs)
        in_uniq_order[order] = vecs                                            # undo the length sort
        return in_uniq_order[inverse].astype(np.float16), time.time() - t0


    def saved_embeddings(source, df):
        """Previously saved embeddings for exactly these records and texts with the same model settings, else None."""
        meta_path, emb_path, man_path = (OUT_DIR / f"{source}_meta.parquet", OUT_DIR / f"{source}_embeddings.npy",
                                         OUT_DIR / "manifest.json")
        if not (REUSE_EMBEDDINGS and meta_path.exists() and emb_path.exists() and man_path.exists()):
            return None
        old = json.loads(man_path.read_text(encoding="utf-8"))
        if (old.get("model"), old.get("prefix"), old.get("max_seq_length")) != (MODEL_NAME, PREFIX, MAX_SEQ_LENGTH):
            return None
        meta = pd.read_parquet(meta_path, columns=["entity_id", "text"])
        if meta["entity_id"].tolist() != df["entity_id"].tolist() or meta["text"].tolist() != df["text"].tolist():
            return None
        return np.load(emb_path), old.get("timing", {}).get(source)


    EMB, TIMING = {}, {}
    META_COLS = ["entity_id", "split", "country", "kind", "business_name", "business_address", "name_norm",
                 "addr_norm", "text"] + TAGS
    for source, df in (("s1", S1), ("s2", S2), ("s3", S3)):
        df["text"] = build_text(df)
        cached = saved_embeddings(source, df)
        if cached is not None:
            EMB[source], TIMING[source] = cached
            print(f"{source}: {EMB[source].shape} loaded from {OUT_DIR} (same records, texts and model)")
            continue
        EMB[source], secs = embed(df["text"].tolist())
        TIMING[source] = {"records": len(df), "seconds": round(secs, 1), "texts_per_sec": round(len(df) / max(secs, 1e-9))}
        extra = ["n_true_matches"] if source == "s1" else ["true_s1"]
        np.save(OUT_DIR / f"{source}_embeddings.npy", EMB[source])
        df[META_COLS + extra].to_parquet(OUT_DIR / f"{source}_meta.parquet", index=False)
        print(f"{source}: {EMB[source].shape} saved in {secs:.0f}s ({TIMING[source]['texts_per_sec']:,} texts/s)")
    stop_pool()

    manifest = {
        "model": MODEL_NAME, "prefix": PREFIX, "text_mode": TEXT_MODE, "max_seq_length": MAX_SEQ_LENGTH,
        "device": GPU_NAME, "gpus": len(GPUS), "fp16": USE_FP16, "batch_size": BATCH_SIZE, "seed": SEED,
        "sample_name": SAMPLE_NAME, "n_by_source": N_BY_SOURCE, "countries": COUNTRIES or "all",
        "bucket_method": BUCKET_METHOD, "top_k": TOP_K,
        "linked_share": LINKED_SHARE, "min_tag_share": MIN_TAG_SHARE, "processed_dir": str(PROCESSED_DIR),
        "counts": {s: {f"{c}/{k}": int(v) for (c, k), v in d.groupby(["country", "kind"]).size().items()}
                   for s, d in (("s1", S1), ("s2", S2), ("s3", S3))},
        "sample_tag_shares": {f"{s}/{c}": r for (s, c), r in SAMPLE_TAGS.to_dict("index").items()},
        "population_tag_shares": POPULATION_TAGS, "timing": TIMING,
    }
    (OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=1, default=str), encoding="utf-8")
    print("saved to", OUT_DIR)

## Quick check: do the embeddings find the true S1?
For every linked S2/S3 record: its cosine to its own S1, its rank among all sampled S1 of the same
country, and the best wrong S1. For no-match records: their best S1 cosine. The numbers are
optimistic, because the S1 index here holds only the sample, not all S1 records.

In [ ]:
def check_step():
    global CHECK, check
    def check(source, df):
        """Recall@k and cosine summaries of one source against the sampled S1 of the same country."""
        rows = []
        for country in sorted(df["country"].unique()):
            s_rows = np.flatnonzero((S1["country"] == country).to_numpy())
            q_rows = np.flatnonzero(((df["country"] == country) & df["kind"].isin(["linked", "no_match"])).to_numpy())
            if not len(s_rows) or not len(q_rows):
                continue
            col_of = {e: j for j, e in enumerate(S1["entity_id"].to_numpy()[s_rows])}
            true_col = np.array([col_of.get(t, -1) for t in df["true_s1"].to_numpy()[q_rows]])
            index = torch.from_numpy(EMB["s1"][s_rows]).to(DEVICE).float()
            rank = np.full(len(q_rows), -1)
            true_cos = np.full(len(q_rows), np.nan, np.float32)
            best_cos = np.full(len(q_rows), np.nan, np.float32)
            for s in range(0, len(q_rows), 2048):
                sims = torch.from_numpy(EMB[source][q_rows[s:s + 2048]]).to(DEVICE).float() @ index.T
                best_cos[s:s + 2048] = sims.max(dim=1).values.cpu().numpy()
                t = torch.from_numpy(true_col[s:s + 2048]).to(DEVICE)
                has = torch.nonzero(t >= 0).flatten()
                if len(has):
                    tc = sims[has, t[has]]
                    true_cos[s + has.cpu().numpy()] = tc.cpu().numpy()
                    rank[s + has.cpu().numpy()] = (sims[has] > tc[:, None]).sum(dim=1).cpu().numpy()
            sub = df.iloc[q_rows]
            linked = true_col >= 0
            slices = {"all linked": linked}
            slices.update({t: linked & sub[t].to_numpy() for t in TAGS})
            for name, m in slices.items():
                if m.sum() < 20:
                    continue
                rows.append({"source": source, "country": country, "slice": name, "n": int(m.sum()),
                             "recall@1": round(float((rank[m] < 1).mean()), 3),
                             "recall@5": round(float((rank[m] < 5).mean()), 3),
                             "recall@10": round(float((rank[m] < 10).mean()), 3),
                             "true cos (median)": round(float(np.median(true_cos[m])), 3)})
            no_match = sub["kind"].to_numpy() == "no_match"
            if no_match.sum():
                rows.append({"source": source, "country": country, "slice": "no-match: best S1 cos (median)",
                             "n": int(no_match.sum()), "true cos (median)": round(float(np.median(best_cos[no_match])), 3)})
        return rows


    CHECK = pd.DataFrame(check("s2", S2) + check("s3", S3))
    CHECK.to_csv(OUT_DIR / "quick_check.csv", index=False)
    display(CHECK)

## A top-k candidate bucket for every S1: exact search and FAISS IVF-PQ

**Goal:** each S1 gets a bucket of `TOP_K` *potential* matches from S2/S3. What matters is that
the true matches are in the bucket (**bucket recall**); reproducing exact nearest-neighbour search
is not the aim. Per country (matches never cross countries), all sampled S2+S3 embeddings are
indexed and every S1 of that country retrieves its bucket.

| Part | Setting | Why |
|---|---|---|
| Mean-centering | subtract the mean S2/S3 vector, re-normalize | e5 vectors share a large common direction (almost all cosines are 0.85–1.0). Removing it spreads them out and raised exact true-match recall@50 from 93.0% to 95.3% (India, 10% sample) |
| IVF | `nlist` ≈ √N clusters; `nprobe` = `NPROBE_FRAC` (25%) of them | a query scans only the nearest clusters. On this data this step is where recall is lost (see below) |
| PQ | 48 sub-vectors × 8 bits = 48 bytes/vector (64 for e5-large, whose 1024 numbers do not divide by 48) | memory: 48 bytes instead of 1,536 (fp32) per vector |
| Exact re-ranking | top `REFINE_FACTOR`·k (10·k) PQ hits re-scored exactly | after re-ranking, PQ cost **no** recall in our tests (IVF-Flat gave identical recall) |

**Measured on the 10% sample** (India, true-match recall in a top-50 bucket): exact search 95.3%,
IVF scanning 25% of clusters 85.6%, 10% 74.8%, 5% 66.8%; HNSW (efSearch 256) 89.3%. Training the
clusters on S1+S2/S3 did not help. `BUCKET_METHOD` picks which buckets are saved as
`s1_buckets.parquet` and scored: `ivfpq` (default) computes both methods and compares them, `exact`
runs exact search only.

**Exact search on the GPUs:**
* Every GPU holds a copy of the country's S2+S3 vectors and takes its share of the S1 queries,
  in blocks sized to keep each score matrix under about 1 GB.
* Each block is scored against all vectors in one matrix product, in fp16 on tensor-core GPUs.
* The best k + 32 per S1 are then re-scored in fp32, so rounding cannot reorder the final top k.
* The original e5 cosine of every bucket pair is also computed on the GPU.
* Without a GPU, FAISS `IndexFlatIP` runs on the CPU and gives the same results.

At full scale (0.7–1.3M S1 against 3.8–4.7M S2+S3 per country), this is on the order of minutes
per country on two T4s. IVF-PQ stays on the CPU (`faiss-cpu`). It saves memory only without its
exact re-ranking, which here keeps a full copy of the vectors. Raise `NPROBE_FRAC` to trade IVF
speed for recall.

In [ ]:
def bucket_step():
    global BUCKETS, BUCKETS_ALL, CANDIDATES, CAND_EMB, FAISS_DIR, METHODS, PQ_M, PQ_NBITS, REFINE_FACTOR
    global RESCORE_EXTRA, SEARCH_DTYPE, ThreadPoolExecutor, b, bucket_frame, build_ivfpq, c_rows, centered
    global country, e_hits, e_scores, exact_search, f, faiss, faiss_stats, frames, hits, importlib, index
    global ivfpq, k, key, m, mu, nlist, pair_cosine, pq_m, q_rows, queries, scores, stats, subprocess, sys, t0
    global t_build, t_search, val, vecs
    import importlib.util
    import subprocess
    import sys
    from concurrent.futures import ThreadPoolExecutor

    if importlib.util.find_spec("faiss") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu"])
    import faiss

    PQ_M, PQ_NBITS = 48, 8                                         # sub-vectors (adjusted to the vector size, see pq_m) x bits
    REFINE_FACTOR = 10                                             # exact re-ranking of the top 10·k PQ hits
    METHODS = ["ivfpq", "exact"] if BUCKET_METHOD == "ivfpq" else ["exact"]
    FAISS_DIR = OUT_DIR / "faiss"
    FAISS_DIR.mkdir(exist_ok=True)


    def centered(vecs, mu):
        """Subtract mu and re-normalize rows (float32, C-contiguous) for FAISS."""
        out = vecs.astype(np.float32) - mu
        out /= np.maximum(np.linalg.norm(out, axis=1, keepdims=True), 1e-12)
        return np.ascontiguousarray(out, dtype=np.float32)


    def pq_m(d):
        """PQ sub-vectors: PQ_M when it divides the vector size (384 for e5-small, 768 for e5-base), else the largest divisor
        of d up to 64 (1024 for e5-large -> 64). FAISS needs d to be a multiple of the number of sub-vectors."""
        return PQ_M if d % PQ_M == 0 else max(m for m in range(1, 65) if d % m == 0)


    def build_ivfpq(vecs):
        """IVF-PQ (inner product) wrapped in exact re-ranking. Returns (index, ivfpq, nlist)."""
        n, d = vecs.shape
        nlist = int(np.clip(np.sqrt(n), 16, max(16, n // 39)))
        quantizer = faiss.IndexFlatIP(d)
        ivfpq = faiss.IndexIVFPQ(quantizer, d, nlist, pq_m(d), PQ_NBITS, faiss.METRIC_INNER_PRODUCT)
        ivfpq.train(vecs)
        index = faiss.IndexRefineFlat(ivfpq)        # keeps full vectors for exact re-ranking
        index.k_factor = REFINE_FACTOR
        index.add(vecs)                             # adds to both the PQ index and the exact store
        ivfpq.nprobe = max(1, round(NPROBE_FRAC * nlist))
        return index, ivfpq, nlist


    SEARCH_DTYPE = torch.float16 if USE_FP16 else torch.float32    # first scoring pass (fp16 = tensor cores)
    RESCORE_EXTRA = 32                                             # the top k + 32 of that pass are re-scored in fp32


    def exact_search(vecs, queries, k):
        """Exact top-k by inner product.

        With GPUs: every GPU holds a copy of the S2/S3 vectors and takes its share of the S1 query blocks
        (one thread per GPU). A block is scored against all vectors in one matrix product, then its best
        k + RESCORE_EXTRA are re-scored in fp32 and the top k kept. Without a GPU: FAISS IndexFlatIP.
        """
        if not GPUS:
            index = faiss.IndexFlatIP(vecs.shape[1])
            index.add(vecs)
            return index.search(queries, k)
        shortlist = min(len(vecs), k + RESCORE_EXTRA)
        rows = int(np.clip(2.5e8 // len(vecs), 64, 4096))          # queries per block: score block <= ~1 GB
        blocks = [(s, min(s + rows, len(queries))) for s in range(0, len(queries), rows)]
        scores = np.empty((len(queries), k), np.float32)
        hits = np.empty((len(queries), k), np.int64)

        def run(device, my_blocks):
            with torch.cuda.device(device):
                base = torch.from_numpy(vecs).to(device, SEARCH_DTYPE)
                for s, e in my_blocks:
                    q = torch.from_numpy(queries[s:e]).to(device)
                    cand = torch.topk(q.to(SEARCH_DTYPE) @ base.T, shortlist, dim=1).indices
                    rescored = torch.einsum("qd,qkd->qk", q, base[cand].float())
                    top = torch.topk(rescored, k, dim=1)
                    scores[s:e] = top.values.cpu().numpy()
                    hits[s:e] = torch.gather(cand, 1, top.indices).cpu().numpy()
                del base
                torch.cuda.empty_cache()

        with ThreadPoolExecutor(len(GPUS)) as ex:
            list(ex.map(run, GPUS, [blocks[i::len(GPUS)] for i in range(len(GPUS))]))
        return scores, hits


    def pair_cosine(a, a_rows, b, b_rows, chunk=250_000):
        """Dot product of a[a_rows[j]] and b[b_rows[j]] for every j (original e5 cosine), in chunks, split
        across all GPUs when there are several (as exact_search does) -- else on the one GPU, else the CPU."""
        out = np.empty(len(a_rows), np.float32)
        if GPUS:
            blocks = [(s, min(s + chunk, len(a_rows))) for s in range(0, len(a_rows), chunk)]

            def run(device, my_blocks):
                with torch.cuda.device(device):
                    ta, tb = torch.from_numpy(a).to(device), torch.from_numpy(b).to(device)
                    for s, e in my_blocks:
                        ia = torch.from_numpy(a_rows[s:e]).to(device)
                        ib = torch.from_numpy(b_rows[s:e]).to(device)
                        out[s:e] = (ta[ia].float() * tb[ib].float()).sum(dim=1).cpu().numpy()
                    del ta, tb
                    torch.cuda.empty_cache()

            with ThreadPoolExecutor(len(GPUS)) as ex:
                list(ex.map(run, GPUS, [blocks[i::len(GPUS)] for i in range(len(GPUS))]))
            return out
        for s in range(0, len(a_rows), chunk):
            out[s:s + chunk] = np.einsum("ij,ij->i", a[a_rows[s:s + chunk]].astype(np.float32),
                                         b[b_rows[s:s + chunk]].astype(np.float32))
        return out


    def bucket_frame(hits, scores, q_rows, c_rows, country, method):
        """Long table of buckets: one row per (S1, candidate) with rank, centered score and original cosine."""
        k = hits.shape[1]
        valid = (hits >= 0).ravel()
        qi = np.repeat(np.arange(len(q_rows)), k)[valid]
        ci = hits.ravel()[valid]
        return pd.DataFrame({
            "s1_entity_id": S1["entity_id"].to_numpy()[q_rows][qi],
            "rank": np.tile(np.arange(k), len(q_rows))[valid],
            "candidate_entity_id": CANDIDATES["entity_id"].to_numpy()[c_rows][ci],
            "candidate_source": CANDIDATES["source"].to_numpy()[c_rows][ci],
            "score": scores.ravel()[valid].astype(np.float32),                          # centered cosine (retrieval order)
            "cosine": pair_cosine(EMB["s1"][q_rows], qi, CAND_EMB[c_rows], ci),        # original e5 cosine
            "country": country, "method": method})


    CANDIDATES = pd.concat([S2.assign(source="S2"), S3.assign(source="S3")], ignore_index=True)
    CAND_EMB = np.vstack([EMB["s2"], EMB["s3"]])                   # float16, row-aligned with CANDIDATES
    del EMB["s2"], EMB["s3"]        # already used by check_step; only EMB["s1"] is read again below
    frames = {m: [] for m in METHODS}
    faiss_stats = []
    for country in sorted(S1["country"].unique()):
        c_rows = np.flatnonzero((CANDIDATES["country"] == country).to_numpy())
        q_rows = np.flatnonzero((S1["country"] == country).to_numpy())
        if len(c_rows) < 1_000:
            print(f"{country}: only {len(c_rows)} S2/S3 records in the sample, no index built "
                  "(France has no S2/S3 files in processed/)")
            continue
        mu = CAND_EMB[c_rows].astype(np.float32).mean(axis=0, keepdims=True)
        vecs, queries = centered(CAND_EMB[c_rows], mu), centered(EMB["s1"][q_rows], mu)
        np.save(FAISS_DIR / f"mean_{country}.npy", mu)             # needed to query with the same centering later
        k = min(TOP_K, len(c_rows))
        stats = {"country": country, "S2+S3 vectors": len(c_rows), "S1 queries": len(q_rows), "k": k}
        if "ivfpq" in METHODS:
            t0 = time.time()
            index, ivfpq, nlist = build_ivfpq(vecs)
            t_build = time.time() - t0
            t0 = time.time()
            scores, hits = index.search(queries, k)
            t_search = time.time() - t0
            frames["ivfpq"].append(bucket_frame(hits, scores, q_rows, c_rows, country, "ivfpq"))
            faiss.write_index(index, str(FAISS_DIR / f"index_{country}.faiss"))
            del index
            stats.update({"nlist": nlist, "nprobe": ivfpq.nprobe, "IVF-PQ build s": round(t_build, 1),
                          "IVF-PQ search s": round(t_search, 1)})
        t0 = time.time()
        e_scores, e_hits = exact_search(vecs, queries, k)
        stats.update({"exact search s": round(time.time() - t0, 1), "exact on": f"{len(GPUS)} GPU" if GPUS else "CPU"})
        frames["exact"].append(bucket_frame(e_hits, e_scores, q_rows, c_rows, country, "exact"))
        faiss_stats.append(stats)
        print(", ".join(f"{key} {val:,}" if isinstance(val, int) else f"{key} {val}" for key, val in stats.items()))

    BUCKETS_ALL = {m: pd.concat(f, ignore_index=True) for m, f in frames.items()}
    del frames                     # else it survives as a global: a second full copy of the bucket table
    for m, b in BUCKETS_ALL.items():
        for col in ("s1_entity_id", "candidate_entity_id", "candidate_source", "country", "method"):
            b[col] = b[col].astype("category")     # repeated strings: several-fold cheaper, same values
        b.to_parquet(FAISS_DIR / f"s1_buckets_{m}.parquet", index=False)
    BUCKETS = BUCKETS_ALL[BUCKET_METHOD]
    BUCKETS.to_parquet(FAISS_DIR / "s1_buckets.parquet", index=False)
    print(f"buckets saved; s1_buckets.parquet = {BUCKET_METHOD}")
    display(pd.DataFrame(faiss_stats))

### How many true matches land in each bucket?
For S1 with at least one true match inside the sample, **bucket recall@k** is the share of those
matches found in the S1's top-k. It is averaged per S1 (like the challenge metric) and pooled over
all pairs, for IVF-PQ and exact search side by side.

In [ ]:
def recall_step():
    global BUCKET_RECALL, LABELED, TRUE_OF, b, country, d, found, g, k, m, method, n_true_all, rows, s, top1
    global v, with_labels
    TRUE_OF = CANDIDATES[CANDIDATES["kind"] == "linked"].groupby("true_s1")["entity_id"].apply(set).to_dict()


    def with_labels(buckets):
        """Add is_true (candidate is a true match of that S1) and is_owner (S1 is the candidate's best-scoring S1)."""
        b = buckets.copy()
        b["is_true"] = [c in TRUE_OF.get(s, ()) for s, c in zip(b["s1_entity_id"], b["candidate_entity_id"])]
        owner = (b.sort_values("cosine", ascending=False).drop_duplicates("candidate_entity_id")
                 .set_index("candidate_entity_id")["s1_entity_id"])
        b["is_owner"] = b["candidate_entity_id"].map(owner).to_numpy() == b["s1_entity_id"].to_numpy()
        return b


    LABELED = {m: with_labels(b) for m, b in BUCKETS_ALL.items()}
    n_true_all = pd.Series({s: len(v) for s, v in TRUE_OF.items()}, name="n_true")
    rows = []
    for method, b in LABELED.items():
        for k in (1, 5, 10, 20, 50):
            if k > TOP_K:
                continue
            found = b[b["is_true"] & (b["rank"] < k)].groupby("s1_entity_id").size()
            d = n_true_all[n_true_all.index.isin(b["s1_entity_id"].unique())].to_frame()
            d["found"] = found.reindex(d.index).fillna(0)
            d["country"] = d.index.map(S1.set_index("entity_id")["country"])
            for country, g in d.groupby("country"):
                rows.append({"country": country, "k": k, "method": method, "S1 with matches": len(g),
                             "bucket recall (mean per S1)": round(float((g["found"] / g["n_true"]).mean()), 4),
                             "bucket recall (all pairs)": round(float(g["found"].sum() / g["n_true"].sum()), 4)})
    BUCKET_RECALL = pd.DataFrame(rows)
    BUCKET_RECALL.to_csv(FAISS_DIR / "bucket_recall.csv", index=False)
    display(BUCKET_RECALL.pivot(index=["country", "k"], columns="method",
                                values=["bucket recall (mean per S1)", "bucket recall (all pairs)"]))

    top1 = LABELED[BUCKET_METHOD][LABELED[BUCKET_METHOD]["rank"] == 0].merge(
        S1[["entity_id", "kind"]], left_on="s1_entity_id", right_on="entity_id")
    print("top-1 original cosine by S1 kind (can a score alone spot singletons?):")
    display(top1.groupby(["country", "kind"])["cosine"].describe(percentiles=[0.1, 0.5, 0.9]).round(3))

### Challenge metric (macro F0.5) on the buckets
Scored over the sampled **labeled** S1: true singletons, plus S1 with at least one true match
inside the sample. "True matches" means the sampled S2/S3 records linked to that S1, since
unsampled ones cannot be retrieved. Rows:
* **Ceiling:** a perfect matcher that keeps exactly the true matches in the top-k bucket. This is
  the best score the buckets allow, shown for IVF-PQ and exact search.
* **Predict nothing / whole top-k:** naive baselines.
* **Cosine ≥ t:** keep bucket entries whose original e5 cosine is above a cutoff. With "one owner",
  each S2/S3 is only kept for the S1 that scores it highest, because every S2/S3 has at most one
  true S1. The cutoff is tuned on this same sample, so treat it as optimistic.

In [ ]:
def metric_step():
    global B, BEST_T, EVAL, EVAL_COUNTRY, EVAL_IDS, EVAL_POS, LABELED, METRIC, N_TRUE, THRESHOLDS, b, best, cos
    global extra, k, kept, label, method, pred, reference, rows, s, score, sweep, t, t_best
    EVAL_IDS = S1.loc[(S1["split"] == "train") & ((S1["kind"] == "singleton") | S1["entity_id"].isin(TRUE_OF)),
                      "entity_id"]
    N_TRUE = pd.Series({s: len(TRUE_OF.get(s, ())) for s in EVAL_IDS}, name="n_true")
    EVAL_COUNTRY = N_TRUE.index.map(S1.set_index("entity_id")["country"]).to_numpy()
    EVAL_POS = pd.Series(np.arange(len(N_TRUE)), index=N_TRUE.index)   # S1 id -> position in N_TRUE


    def score(b, mask):
        """Challenge macro F0.5 when the rows of bucket table b selected by mask are the predictions (vectorized)."""
        pos = b["s1_pos"].to_numpy()[mask]
        n_pred = np.bincount(pos, minlength=len(N_TRUE))
        tp = np.bincount(pos, weights=b["is_true"].to_numpy()[mask].astype(float), minlength=len(N_TRUE))
        n_true = N_TRUE.to_numpy()
        with np.errstate(divide="ignore", invalid="ignore"):
            p, r = tp / n_pred, tp / n_true
            f = np.where(tp > 0, 1.25 * p * r / (0.25 * p + r), 0.0)
        f = np.where(n_true == 0, (n_pred == 0).astype(float), f)
        out = {"macro F0.5": f.mean(), "S1 with matches": f[n_true > 0].mean(), "singletons": f[n_true == 0].mean()}
        out.update({f"macro F0.5 {c}": f[EVAL_COUNTRY == c].mean() for c in sorted(set(EVAL_COUNTRY))})
        return out


    EVAL = {}
    for method, b in LABELED.items():
        EVAL[method] = b[b["s1_entity_id"].isin(N_TRUE.index)].copy()
        EVAL[method]["s1_pos"] = EVAL[method]["s1_entity_id"].map(EVAL_POS).astype(np.int64).to_numpy()
    del LABELED                    # fully consumed above; recall_step rebuilds it fresh for the next model
    B = EVAL[BUCKET_METHOD]
    rows = [{"method": "predict nothing", "buckets": "-", "k": 0, "cosine ≥": None, **score(B, np.zeros(len(B), bool))}]
    for method, b in EVAL.items():
        for k in (1, 5, 10, 20, 50):
            if k <= TOP_K:
                rows.append({"method": "ceiling: perfect matcher on the top-k bucket", "buckets": method, "k": k,
                             "cosine ≥": None, **score(b, (b["is_true"] & (b["rank"] < k)).to_numpy())})
    for k in (1, 5):
        rows.append({"method": "predict the whole top-k bucket", "buckets": BUCKET_METHOD, "k": k, "cosine ≥": None,
                     **score(B, (B["rank"] < k).to_numpy())})
    THRESHOLDS = np.round(np.arange(0.80, 0.995, 0.005), 3)
    cos = B["cosine"].to_numpy()
    BEST_T = {}
    for label, extra in [("cosine ≥ t", np.ones(len(B), bool)),
                         ("cosine ≥ t, one owner per S2/S3", B["is_owner"].to_numpy())]:
        sweep = [(t, score(B, (cos >= t) & extra)) for t in THRESHOLDS]
        t_best, best = max(sweep, key=lambda x: x[1]["macro F0.5"])
        BEST_T[label] = t_best
        rows.append({"method": label + " (t tuned on this sample)", "buckets": BUCKET_METHOD, "k": TOP_K,
                     "cosine ≥": t_best, **best})
    METRIC = pd.DataFrame(rows)
    METRIC.to_csv(FAISS_DIR / "metric_f05.csv", index=False)

    # cross-check the vectorized score against the reference f05() for the best one-owner cutoff
    t = BEST_T["cosine ≥ t, one owner per S2/S3"]
    kept = B[(cos >= t) & B["is_owner"].to_numpy()]
    pred = kept.groupby("s1_entity_id")["candidate_entity_id"].apply(list).to_dict()
    reference = macro_f05(pred, TRUE_OF, N_TRUE.index)
    assert abs(reference - METRIC.iloc[-1]["macro F0.5"]) < 1e-9, (reference, METRIC.iloc[-1]["macro F0.5"])
    print(f"{len(N_TRUE):,} labeled S1 scored ({int((N_TRUE == 0).sum()):,} singletons); "
          f"vectorized score matches the reference metric")
    display(METRIC.round(4))
    del EVAL, B                    # not read by extra_step or the next model's steps; metric_step rebuilds them

## Word map for Indian-script names
As in 02 (full): most Indian-script business names are English words written in that script, so their
transliterations are misspelled English ("smart solyusns praivet limited"). A map learned from the
train true pairs of the full `processed/` data sends them back to English ("smart solutions private
limited"). It is used for the names in the e5 text (`TEXT_MODE = "fixed"`) and in the name search below,
learned once (about 1-2 minutes) and saved as `embeddings/translit_wordmap.json`. `WORD_MAP = false` turns
it off.

In [ ]:
import collections  # noqa: E402
import importlib.util  # noqa: E402
import subprocess  # noqa: E402
import sys  # noqa: E402
if importlib.util.find_spec("rapidfuzz") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rapidfuzz"])
from rapidfuzz import fuzz  # noqa: E402  (MIT)
from rapidfuzz import process as rf_process  # noqa: E402

# Word map for Indian scripts: most Indian-script business names are English words written in that script, and
# their transliterations are misspelled English ("praivet limited", "solyusns", "injiniyring"). The map sends such
# words back to English ("private limited", "solutions", "engineering"). It is learned from the train true pairs
# whose candidate name is in an Indian script: each transliterated word is lined up with the most similar word of
# its true S1's name (rapidfuzz ratio >= 55), and kept when the same English word wins in at least
# WORDMAP_MIN_PAIRS pairs and WORDMAP_MIN_SHARE of the word's occurrences. Test bed (map learned without its S1):
# name similarity of Indian-script true pairs 72-77 -> 88-92; matcher 0.9667 -> 0.9675, country transfer +0.003-0.005.
WORDMAP_MIN_PAIRS, WORDMAP_MIN_SHARE, WORDMAP_MIN_RATIO = 20, 0.6, 55
_RE_WORD2 = re.compile(r"[a-z]{2,}")


def learn_word_map(processed_dir, gt_file):
    """{transliterated word: English word} from the train true pairs of every country (only Indian-script names count)."""
    gt = pd.read_csv(gt_file, sep="\t", dtype=object, keep_default_na=False, na_filter=False, quoting=csv.QUOTE_NONE)
    gp = gt[gt["matched_entity_ids"] != ""]
    gp = gp.assign(cand=gp["matched_entity_ids"].str.split(",")).explode("cand")
    owner = dict(zip(gp["cand"], gp["source1_entity_id"]))
    del gt, gp
    s1_name = {}
    for f in sorted(Path(processed_dir).glob("train_s1_*.parquet")):
        d = pd.read_parquet(f, columns=["entity_id", "business_name"])
        s1_name.update(zip(d["entity_id"], d["business_name"].fillna("").astype(str)))
    pair_counts, tok_counts = collections.Counter(), collections.Counter()
    for f in sorted(list(Path(processed_dir).glob("train_s2_*.parquet")) + list(Path(processed_dir).glob("train_s3_*.parquet"))):
        d = pd.read_parquet(f, columns=["entity_id", "business_name"])
        for e, name in zip(d["entity_id"], d["business_name"].fillna("").astype(str)):
            s = owner.get(e)
            if s is None or s not in s1_name or not RE_NATIVE.search(name):
                continue
            a = set(_RE_WORD2.findall(s1_name[s].lower()))
            for t in set(_RE_WORD2.findall(translit(name).lower())):
                tok_counts[t] += 1
                if t in a or not a:
                    continue
                best = rf_process.extractOne(t, a, scorer=fuzz.ratio, score_cutoff=WORDMAP_MIN_RATIO)
                if best:
                    pair_counts[(t, best[0])] += 1
    best_of = {}
    for (t, e), n in pair_counts.items():
        if n > best_of.get(t, (0, ""))[0] or (n == best_of[t][0] and e < best_of[t][1]):
            best_of[t] = (n, e)
    return {t: e for t, (n, e) in sorted(best_of.items())
            if n >= WORDMAP_MIN_PAIRS and n / tok_counts[t] >= WORDMAP_MIN_SHARE}


def translit_name(s):
    """Transliterated Indian-script name with the word map applied ('प्राइवेट लिमिटेड' -> 'private limited')."""
    return " ".join(WORDMAP.get(w, w) for w in translit(s).lower().split())


WORDMAP = None


def ensure_word_map():
    """Load or learn the word map once (after translit() is defined by the embed step)."""
    global WORDMAP
    if WORDMAP is not None:
        return
    f = EMB_ROOT / "translit_wordmap.json"
    if not WORD_MAP:
        WORDMAP = {}
    elif f.exists():
        WORDMAP = json.loads(f.read_text(encoding="utf-8"))
    else:
        t0 = time.time()
        WORDMAP = learn_word_map(PROCESSED_DIR, GT_FILE)
        f.write_text(json.dumps(WORDMAP, ensure_ascii=False, indent=0, sort_keys=True), encoding="utf-8")
        print(f"word map learned from the train pairs in {time.time() - t0:.0f}s")
    print(f"word map: {len(WORDMAP)} words, e.g. " + ", ".join(f"{k} -> {v}" for k, v in list(WORDMAP.items())[:10]))

## Extra candidate searches (as in 02 full)
e5 compares name and address together, so it misses some matches. 02 (full) adds five searches to each
S1's bucket; here they run for every model on the same sample, so the models can be compared with
the candidates the matcher really judges:

| Search | A pair is added when |
|---|---|
| reverse | the S1 is among the `REVERSE_TOP` (2) best S1 of the S2/S3 record, by this model's centered score |
| address | same house number and first street word (keys shared by more than 20 candidates skipped) |
| name | among the S1's 5 most similar names by character 3-grams (TF-IDF, 3-grams in more than 2% of names ignored) |
| number | (`NUMBER_COUNTRIES`, India by default) the addresses share a number (house, plot, door, PIN) with another word of 3+ letters, e.g. `206 + pune` (keys shared by more than `NUMBER_KEY_CAP` = 10 candidates skipped) |
| empty | the name search over only the candidates with an empty address: each such candidate's `EMPTY_NAME_TOP` (3) closest S1 names, and each S1's `EMPTY_S1_TOP` (2) closest names among them |

The table (`faiss/extra_recall.csv`) has, per country: e5 top-20/30/40/50 alone, e5 top-30 plus each
search, and plus all of them; pairs per S1, the share of the sample's true matches inside (overall, for
Indian-script / Latin-script candidate names, and for candidates with an empty address), and the ceiling
(best macro F0.5 a perfect matcher could reach, over the scored S1 of the metric section).
`EXTRA_SEARCHES = false` skips this step. "all extra" includes number and empty, so it is not comparable
with an "all extra" row saved before they were added.

The step also saves `faiss/s1_buckets_all.parquet` (`SAVE_ALL_BUCKETS`, default on): e5 top-30 plus every
search, in 02 full's bucket format, so that `03_lightgbm_matcher.ipynb` (`BUCKETS=all`, or `sub2` for
submission 2's searches only) gives the real LightGBM macro F0.5 of these candidates on this sample.
It is written one country at a time (about 36 pairs per S1).

In [ ]:
import unicodedata  # noqa: E402

_ACCENTS = {}
for _cp in range(0xC0, 0x250):
    _base = unicodedata.normalize("NFKD", chr(_cp))[0]
    if _base.isascii() and _base != chr(_cp):
        _ACCENTS[_cp] = _base
FOLD = str.maketrans(_ACCENTS)
RE_WORD = re.compile(r"\w+")
STREET_TYPES = frozenset("rue avenue av ave boulevard bd blvd boul chemin che place pl route rte allee impasse imp quai "
                         "cours crs square faubourg fbg road rd street st lane ln drive dr way court ct circle cir "
                         "highway hwy parkway pkwy terrace marg".split())
STREET_STOP = STREET_TYPES | frozenset("r ch all sq bis ter de du des la le les l d of the and et en sur suite ste unit "
                                       "apt apartment building bldg floor fl po box door no nr n number h hno flat shop "
                                       "plot near opp".split())


def fold_text(s):
    """Lower case with Latin accents removed (other scripts unchanged)."""
    return s.lower() if s.isascii() else s.lower().translate(FOLD)


def words(s):
    """Folded word tokens; numbers without leading zeros ('0058' -> '58')."""
    return [(w.lstrip("0") or "0") if w.isdigit() else w for w in RE_WORD.findall(fold_text(s))]


def street_of(address):
    """(street name, house number): the first comma part with a number or a street word, minus numbers, street
    words and stop words. '15 bis Rue de Barbieux, Roubaix' -> ('barbieux', '15')."""
    house = ""
    for part in address.split(","):
        w = words(part)
        nums = [x for x in w if x.isdigit()]
        if nums and not house:
            house = nums[0]
        if nums or any(x in STREET_TYPES for x in w):
            name = " ".join(x for x in w if not x.isdigit() and x not in STREET_STOP and len(x) > 1)
            if name:
                return name, house
    return "", house


REVERSE_K, REVERSE_TOP, ADDRESS_KEY_CAP, NAME_TOP, NAME_MAX_DF = 5, 2, 20, 5, 0.02
NUMBER_KEY_CAP = int(os.environ.get("NUMBER_KEY_CAP", 10))     # as 02 full
NUMBER_COUNTRIES = [c for c in os.environ.get("NUMBER_COUNTRIES", "India").split(",") if c]
EMPTY_NAME_TOP = int(os.environ.get("EMPTY_NAME_TOP", 3))
EMPTY_S1_TOP = int(os.environ.get("EMPTY_S1_TOP", 2))
if EXTRA_SEARCHES and importlib.util.find_spec("polars") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "polars"])
SAVE_ALL_BUCKETS = os.environ.get("SAVE_ALL_BUCKETS", "true") == "true"   # faiss/s1_buckets_all.parquet for 03 (BUCKETS=all)
ALL_E5_TOP = 30                  # e5 candidates in that file, as 02 full's TOP_K and 03 full's CANDIDATES_PER_S1


def address_pairs(s1_addr, c_addr):
    """(S1 row, candidate row) with the same house number and first street word."""
    def keys(addrs):
        codes, uniq = pd.factorize(np.asarray(addrs, dtype=object))
        k = []
        for a in uniq:
            street, house = street_of(a)
            k.append(f"{house}|{street.split()[0]}" if house and street else "")
        return np.asarray(k, dtype=object)[codes]
    left = pd.DataFrame({"q": np.arange(len(s1_addr)), "key": keys(s1_addr)})
    right = pd.DataFrame({"c": np.arange(len(c_addr)), "key": keys(c_addr)})
    left, right = left[left["key"] != ""], right[right["key"] != ""]
    right = right[right["key"].map(right["key"].value_counts()) <= ADDRESS_KEY_CAP]
    j = left.merge(right, on="key")
    return j["q"].to_numpy(np.int64), j["c"].to_numpy(np.int64)


def _csr_to(m, device):
    return torch.sparse_csr_tensor(torch.from_numpy(m.indptr.astype(np.int64)), torch.from_numpy(m.indices.astype(np.int64)),
                                   torch.from_numpy(m.data), size=m.shape, device=device)


def _top_per_row(r, c, v, t):
    """Keep the t best (c, v) per row r; returns (row, col, rank)."""
    order = torch.argsort(-v, stable=True)
    order = order[torch.argsort(r[order], stable=True)]              # grouped by row, best first
    r, c = r[order], c[order]
    pos = torch.arange(len(r), device=r.device) - torch.searchsorted(r, r)
    keep = pos < t
    return r[keep].cpu().numpy(), c[keep].cpu().numpy(), pos[keep].cpu().numpy()


def name_vectorizer():
    """Character 3-gram TF-IDF of the name searches (name and empty)."""
    from sklearn.feature_extraction.text import TfidfVectorizer
    return TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 3), min_df=2, max_df=NAME_MAX_DF, sublinear_tf=True,
                           dtype=np.float32)


def name_pairs(s1_names, c_names, rows=256):
    """(S1 row, candidate row, rank) of the NAME_TOP candidates per S1 with the most similar name
    (cosine of character 3-gram TF-IDF vectors), with sparse matrix products on the GPUs."""
    vec = name_vectorizer()
    try:
        vec.fit(np.concatenate([s1_names, c_names]))
    except ValueError:                                               # no 3-gram left (tiny inputs)
        return (np.empty(0, np.int64),) * 3
    a, bt = vec.transform(s1_names).tocsr(), vec.transform(c_names).T.tocsr()     # S1 x V, V x candidates
    return sparse_top(a, bt, NAME_TOP, rows)


def empty_pairs(s1_names, c_names, c_empty):
    """The name search over only the candidates with an empty address, both ways: (S1 row, candidate row, rank) of
    each S1's EMPTY_S1_TOP closest names among them, and of each such candidate's EMPTY_NAME_TOP closest S1 names."""
    none = (np.empty(0, np.int64),) * 3
    pool = np.flatnonzero(c_empty)
    if not len(pool) or not len(s1_names):
        return none, none
    vec = name_vectorizer()
    try:
        vec.fit(np.concatenate([s1_names, c_names[pool]]))
    except ValueError:
        return none, none
    a, p = vec.transform(s1_names).tocsr(), vec.transform(c_names[pool]).tocsr()    # S1 x V, pool x V
    fwd, rev = none, none
    if EMPTY_S1_TOP:
        q, c, r = sparse_top(a, p.T.tocsr(), EMPTY_S1_TOP)
        fwd = (q, pool[c], r)
    if EMPTY_NAME_TOP:
        c, q, r = sparse_top(p, a.T.tocsr(), EMPTY_NAME_TOP)
        rev = (q, pool[c], r)
    return fwd, rev


NUMBER_STOP = sorted(STREET_STOP | {"null", "nan", "none"})


def number_keys(addrs):
    """(row, key) for every (number, word) combination of each address: any number (letters attached, leading zeros
    dropped: '0045A' -> '45a') with any other word of 3+ letters that is not a street or stop word."""
    import polars as pl
    keep = pl.element().filter((pl.element().str.len_chars() >= 3) & ~pl.element().is_in(NUMBER_STOP))
    df = pl.DataFrame({"a": [fold_text(x) for x in addrs]}, schema={"a": pl.String}).with_row_index("r")
    df = df.select("r", nums=pl.col("a").str.extract_all(r"[0-9a-z]*[0-9][0-9a-z]*")
                   .list.eval(pl.element().str.replace(r"^0+([0-9a-z])", "${1}")).list.unique(),
                   words=pl.col("a").str.extract_all(r"[a-z]+").list.eval(keep).list.unique())
    return (df.explode("nums").explode("words").drop_nulls()
            .select("r", key=pl.concat_str(["nums", "words"], separator="|").hash(seed=0)))


def number_pairs(s1_addr, c_addr):
    """(S1 row, candidate row) whose addresses share a (number, word) key; keys shared by more than
    NUMBER_KEY_CAP candidates are skipped."""
    import polars as pl
    ck = number_keys(c_addr)
    ck = ck.join(ck.group_by("key").len().filter(pl.col("len") <= NUMBER_KEY_CAP).select("key"), on="key", how="semi")
    j = number_keys(s1_addr).join(ck, on="key", how="inner").select(q="r", c="r_right").unique()
    return j["q"].to_numpy().astype(np.int64), j["c"].to_numpy().astype(np.int64)


def attach_values(all_keys, keys, values, fill=np.nan):
    """values of keys at their position in all_keys (sorted, unique); fill elsewhere."""
    out = np.full(len(all_keys), fill, np.float32)
    if len(keys):
        pos = np.minimum(np.searchsorted(all_keys, keys), len(all_keys) - 1)
        ok = all_keys[pos] == keys
        out[pos[ok]] = np.asarray(values, np.float32)[ok] if np.ndim(values) else values
    return out


SOURCE_COLUMNS = (("reverse", "rev_rank", np.nan), ("address", "addr_key", 0.0), ("name", "name_rank", np.nan),
                  ("number", "num_key", 0.0), ("empty", "empty_rank", np.nan), ("empty_rev", "empty_rev_rank", np.nan))


def all_bucket_frame(country, s1_ids, c_ids, s1_vecs, c_vecs, queries, vecs, e5_q, e5_c, e5_r, sources):
    """One country's bucket as 02 full saves it: the e5 top-ALL_E5_TOP plus every extra search's pairs, with rank (by
    centered score), centered score, original cosine and which searches found each pair. sources:
    {search: (S1 rows, candidate rows, value)}, value = rank (1 for the key searches); rows are within the country."""
    n_c = len(c_ids)
    key = lambda q, c: np.asarray(q, np.int64) * n_c + np.asarray(c, np.int64)  # noqa: E731
    e5_keys = key(e5_q, e5_c)
    added = [e5_keys]
    for name, (q, c, v) in sources.items():
        k = key(q, c)
        added.append(k[np.asarray(v) < REVERSE_TOP] if name == "reverse" else k)   # reverse: the candidate's best 2 S1
    all_keys = np.unique(np.concatenate(added))
    qi, ci = all_keys // n_c, all_keys % n_c
    score = pair_cosine(queries, qi, vecs, ci)                                     # centered cosine (retrieval order)
    cosine = pair_cosine(s1_vecs, qi, c_vecs, ci)                                  # original e5 cosine
    cols = {"e5_rank": attach_values(all_keys, e5_keys, e5_r)}
    for name, col, fill in SOURCE_COLUMNS:
        q, c, v = sources.get(name, ((), (), 0))
        cols[col] = attach_values(all_keys, key(q, c), v, fill)
    order = np.lexsort((-score, qi))                                              # grouped by S1, best score first
    qi, ci = qi[order], ci[order]
    starts = np.r_[0, np.flatnonzero(qi[1:] != qi[:-1]) + 1]
    rank = np.arange(len(qi)) - np.repeat(starts, np.diff(np.r_[starts, len(qi)]))
    df = pd.DataFrame({"s1_entity_id": s1_ids[qi], "rank": rank.astype(np.int16), "candidate_entity_id": c_ids[ci]})
    df["candidate_source"] = df["candidate_entity_id"].str[:2]
    df["score"], df["cosine"], df["country"] = score[order], cosine[order], country
    for col, v in cols.items():
        df[col] = v[order].astype(np.int8) if col in ("addr_key", "num_key") else v[order]
    return df


def sparse_top(a, bt, top, rows=256):
    """(row of a, column of bt, rank) of the `top` best entries of every row of a @ bt (both sparse), on the GPUs
    when there are any (a block of rows that does not fit in GPU memory is split in two)."""
    blocks = [(s, min(s + rows, a.shape[0])) for s in range(0, a.shape[0], rows)]
    if not GPUS:                                                     # CPU (small tests only)
        out = []
        for s, e in blocks:
            p = (a[s:e] @ bt).tocoo()
            r, c, k = _top_per_row(torch.from_numpy(p.row.astype(np.int64)), torch.from_numpy(p.col.astype(np.int64)),
                                   torch.from_numpy(p.data), top)
            out.append((r + s, c, k))
        return tuple(np.concatenate(x) for x in zip(*out)) if out else (np.empty(0, np.int64),) * 3

    def run(device, my_blocks):
        out = []
        with torch.cuda.device(device):
            bt_g = _csr_to(bt, device)
            todo = list(my_blocks)
            while todo:
                s, e = todo.pop()
                try:
                    p = (_csr_to(a[s:e], device) @ bt_g).to_sparse_coo().coalesce()
                    r, c = p.indices()
                    r, c, k = _top_per_row(r, c, p.values(), top)
                    out.append((r + s, c, k))
                    del p
                except (torch.cuda.OutOfMemoryError, RuntimeError) as err:   # too big for the GPU: split the block
                    if "memory" not in str(err).lower() and "cusparse" not in str(err).lower():
                        raise
                    torch.cuda.empty_cache()
                    assert e - s > 1, "a single name block does not fit on the GPU"
                    m = (s + e) // 2
                    todo += [(s, m), (m, e)]
            del bt_g
            torch.cuda.empty_cache()
        return out

    with ThreadPoolExecutor(len(GPUS)) as ex:
        parts = [x for out in ex.map(run, GPUS, [blocks[i::len(GPUS)] for i in range(len(GPUS))]) for x in out]
    return tuple(np.concatenate(x) for x in zip(*parts)) if parts else (np.empty(0, np.int64),) * 3



def extra_step():
    """The extra searches for this model and the recall table of e5 top-k vs e5 top-30 + searches."""
    global EXTRA_RECALL
    if not EXTRA_SEARCHES:
        return
    ensure_word_map()
    e5 = BUCKETS_ALL["exact"]
    all_path, writer = FAISS_DIR / "s1_buckets_all.parquet", None
    import pyarrow as pa
    truth = CANDIDATES.loc[CANDIDATES["kind"] == "linked", ["true_s1", "entity_id", "tag_native_script", "tag_empty_address"]]
    truth = truth.rename(columns={"true_s1": "s1", "entity_id": "cand"})
    rows = []
    for country in sorted(S1["country"].unique()):
        c_rows = np.flatnonzero((CANDIDATES["country"] == country).to_numpy())
        q_rows = np.flatnonzero((S1["country"] == country).to_numpy())
        if len(c_rows) < 1_000:
            continue
        s1_ids = S1["entity_id"].to_numpy()[q_rows]
        c_ids = CANDIDATES["entity_id"].to_numpy()[c_rows]
        mu = CAND_EMB[c_rows].astype(np.float32).mean(axis=0, keepdims=True)
        vecs, queries = centered(CAND_EMB[c_rows], mu), centered(EMB["s1"][q_rows], mu)
        kr = min(REVERSE_K, len(q_rows))
        _, rhits = exact_search(queries, vecs, kr)                      # the best S1 of every S2/S3 record
        rev = pd.DataFrame({"s1": s1_ids[rhits.ravel()], "cand": np.repeat(c_ids, kr), "rank": np.tile(np.arange(kr), len(c_ids))})
        lat = lambda x: translit(x) if RE_NATIVE.search(x) else x  # noqa: E731
        s1_addr = [lat(a) for a in S1["business_address"].fillna("").astype(str).to_numpy()[q_rows]]
        c_addr = [lat(a) for a in CANDIDATES["business_address"].fillna("").astype(str).to_numpy()[c_rows]]
        q, c = address_pairs(s1_addr, c_addr)
        adr = pd.DataFrame({"s1": s1_ids[q], "cand": c_ids[c]})
        sources = {"reverse": (rhits.ravel(), np.repeat(np.arange(len(c_ids)), kr), np.tile(np.arange(kr), len(c_ids))),
                   "address": (q, c, 1.0)}
        nm = lambda r, cn: translit_name(r) if RE_NATIVE.search(r) else cn.lower()  # noqa: E731
        s1_names = np.asarray([nm(r, cn) for r, cn in zip(S1["business_name"].fillna("").astype(str).to_numpy()[q_rows],
                                                             S1["core_name"].fillna("").astype(str).to_numpy()[q_rows])], dtype=object)
        c_names = np.asarray([nm(r, cn) for r, cn in zip(CANDIDATES["business_name"].fillna("").astype(str).to_numpy()[c_rows],
                                                            CANDIDATES["core_name"].fillna("").astype(str).to_numpy()[c_rows])], dtype=object)
        q, c, r = name_pairs(s1_names, c_names)
        nam = pd.DataFrame({"s1": s1_ids[q], "cand": c_ids[c]})
        sources["name"] = (q, c, r)
        fwd, back = empty_pairs(s1_names, c_names, CANDIDATES["tag_empty_address"].to_numpy()[c_rows].astype(bool))
        emp = pd.DataFrame({"s1": s1_ids[np.r_[fwd[0], back[0]].astype(np.int64)],
                            "cand": c_ids[np.r_[fwd[1], back[1]].astype(np.int64)]})
        sources["empty"], sources["empty_rev"] = fwd, back
        num = None
        if country in NUMBER_COUNTRIES:
            q, c = number_pairs(s1_addr, c_addr)
            num = pd.DataFrame({"s1": s1_ids[q], "cand": c_ids[c]})
            sources["number"] = (q, c, 1.0)
        if SAVE_ALL_BUCKETS:                            # the country's bucket as 02 full saves it, for 03 (BUCKETS=all)
            top = e5[(e5["country"] == country) & (e5["rank"] < ALL_E5_TOP)]
            e5_q = pd.Index(s1_ids).get_indexer(top["s1_entity_id"].to_numpy())
            e5_c = pd.Index(c_ids).get_indexer(top["candidate_entity_id"].to_numpy())
            assert (e5_q >= 0).all() and (e5_c >= 0).all(), "e5 bucket rows outside this country's records"
            frame = all_bucket_frame(country, s1_ids, c_ids, EMB["s1"][q_rows], CAND_EMB[c_rows], queries, vecs,
                                     e5_q, e5_c, top["rank"].to_numpy(), sources)
            table = pa.Table.from_pandas(frame, preserve_index=False)
            writer = writer or pq.ParquetWriter(all_path, table.schema)
            writer.write_table(table)
            print(f"{country}: {len(frame) / len(q_rows):.1f} pairs per S1 saved to {all_path.name} "
                  f"(e5 top-{ALL_E5_TOP} + every extra search)")
            del frame, table, top
        eb = e5.loc[e5["country"] == country, ["s1_entity_id", "candidate_entity_id", "rank"]].rename(
            columns={"s1_entity_id": "s1", "candidate_entity_id": "cand"})
        top30 = eb.loc[eb["rank"] < 30, ["s1", "cand"]]
        sets = {f"e5 top-{k}": eb.loc[eb["rank"] < k, ["s1", "cand"]] for k in (20, 30, 40, 50) if k <= TOP_K}
        sets["e5 top-30 + reverse"] = pd.concat([top30, rev.loc[rev["rank"] < REVERSE_TOP, ["s1", "cand"]]])
        sets["e5 top-30 + address"] = pd.concat([top30, adr])
        sets["e5 top-30 + name"] = pd.concat([top30, nam])
        if num is not None:
            sets["e5 top-30 + number"] = pd.concat([top30, num])
        sets["e5 top-30 + empty"] = pd.concat([top30, emp])
        sets["e5 top-30 + all extra"] = pd.concat([top30, rev.loc[rev["rank"] < REVERSE_TOP, ["s1", "cand"]], adr, nam, emp]
                                                  + ([num] if num is not None else []))
        s1_set = set(s1_ids)
        t_c = truth[truth["s1"].isin(s1_set)]
        eval_c = [x for x in EVAL_IDS if x in s1_set]
        n_true = N_TRUE.reindex(eval_c).to_numpy()
        for label, pairs in sets.items():
            pairs = pairs.drop_duplicates()
            hit = t_c.merge(pairs, on=["s1", "cand"])
            nat = t_c["tag_native_script"].astype(bool)
            emp_t = t_c["tag_empty_address"].astype(bool)
            found = hit.groupby("s1").size().reindex(eval_c).fillna(0).to_numpy()
            with np.errstate(divide="ignore", invalid="ignore"):
                rr = found / n_true
                f = np.where(n_true == 0, 1.0, np.where(found > 0, 1.25 * rr / (0.25 + rr), 0.0))
            rows.append({"country": country, "candidates": label, "pairs per S1": round(len(pairs) / len(q_rows), 1),
                         "true matches in bucket": round(len(hit) / max(len(t_c), 1), 4),
                         "... Indian-script candidates": round(hit["tag_native_script"].astype(bool).sum() / nat.sum(), 4) if nat.sum() else np.nan,
                         "... Latin-script candidates": round((~hit["tag_native_script"].astype(bool)).sum() / (~nat).sum(), 4) if (~nat).sum() else np.nan,
                         "... empty-address candidates": round(hit["tag_empty_address"].astype(bool).sum() / emp_t.sum(), 4) if emp_t.sum() else np.nan,
                         "ceiling macro F0.5": round(float(f.mean()), 4) if len(f) else np.nan})
    if writer is not None:
        writer.close()
    EXTRA_RECALL = pd.DataFrame(rows)
    EXTRA_RECALL.to_csv(FAISS_DIR / "extra_recall.csv", index=False)
    print(f"extra searches ({MODEL_NAME}): e5 top-k vs e5 top-30 + reverse / address / name / number / empty")
    display(EXTRA_RECALL)

## Run every model
The sections above are run once per model in `E5_MODELS`, in order: embed (or load the saved
embeddings), quick check, buckets, bucket recall, metric and the extra searches. Each model writes to its own folder
`embeddings/<model>/<sample>/`, so all of them are kept in this run's output, and the last section
compares them. The viewer below shows the last model's buckets.

In [ ]:
import gc  # noqa: E402

RUN_LOG = []
for _name in MODEL_LIST:
    t_model = time.time()
    set_model(_name)
    print(f"\n==================== {MODEL_NAME} -> {OUT_DIR}")
    for step in (embed_step, check_step, bucket_step, recall_step, metric_step, extra_step):
        step()
    RUN_LOG.append({"model": MODEL_NAME, "minutes": round((time.time() - t_model) / 60, 1), "output": str(OUT_DIR)})
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
pd.DataFrame(RUN_LOG)

### Look at the bucket of a particular S1
`show_bucket(s1_entity_id, k)` prints the S1 record and its top-k candidates, marking true
matches. With no argument it picks a random S1 that has matches in the sample.

In [ ]:
S1_BY_ID = S1.set_index("entity_id")
CAND_BY_ID = CANDIDATES.set_index("entity_id")


def show_bucket(s1_entity_id=None, k=10, seed=None):
    """Print one S1 record and its top-k FAISS bucket; ✓ marks a true match."""
    if s1_entity_id is None:
        with_matches = [s for s in BUCKETS["s1_entity_id"].unique() if s in TRUE_OF]
        s1_entity_id = np.random.default_rng(seed).choice(with_matches)
    s1 = S1_BY_ID.loc[s1_entity_id]
    truth = TRUE_OF.get(s1_entity_id, set())
    print(f"S1 {s1_entity_id} [{s1['country']}, {s1['kind']}]: {s1['business_name']} | {s1['business_address']}")
    print(f"true matches in the sample: {len(truth)}; found in top-{k}: "
          f"{len(truth & set(BUCKETS[(BUCKETS['s1_entity_id'] == s1_entity_id) & (BUCKETS['rank'] < k)]['candidate_entity_id']))}")
    b = BUCKETS[(BUCKETS["s1_entity_id"] == s1_entity_id) & (BUCKETS["rank"] < k)].sort_values("rank")
    out = b.assign(match=b["candidate_entity_id"].map(lambda e: "✓" if e in truth else ""),
                   name=b["candidate_entity_id"].map(CAND_BY_ID["business_name"]),
                   address=b["candidate_entity_id"].map(CAND_BY_ID["business_address"]),
                   kind=b["candidate_entity_id"].map(CAND_BY_ID["kind"]))
    # rank follows the centered retrieval score; cosine is the original e5 cosine
    return out[["rank", "score", "cosine", "match", "candidate_entity_id", "candidate_source", "kind", "name", "address"]]


show_bucket(seed=0)

## Compare e5 models on this sample
Every model run on the same sample (same `SAMPLE_NAME` folder under `embeddings/<model>/`), side by side:

1. **Recall:** share of true matches inside the exact-search top-k buckets, per country.
2. **Recall, Indian-script vs Latin-script candidates** (India): where a bigger model or better
   transliteration should show.
3. **Ceiling:** the best macro F0.5 a perfect matcher could reach on the top-k buckets.
4. **Cost:** encoding speed (texts per second) and time per source.
5. **With the extra searches** (as 02 full): e5 top-k alone vs e5 top-30 + reverse / address / name /
   all, per country: pairs per S1, true matches inside (overall and by script), ceiling. This is the
   comparison that matters for the pipeline, since the matcher judges e5 top-30 + all extra.

All models of `E5_MODELS` ran above; models from earlier runs with the same sample settings are
included too. The samples are checked to hold the same S1 records before they are compared, and the
comparison is saved as `embeddings/model_comparison_<sample>.csv`.

In [ ]:
runs, ceilings, timing, scripts = {}, {}, [], {}
native_s1 = None
for f in sorted(EMB_ROOT.glob(f"*/{SAMPLE_NAME}/faiss/bucket_recall.csv")):
    run_dir = f.parent.parent
    model = run_dir.parent.name
    ids = pd.read_parquet(run_dir / "s1_meta.parquet", columns=["entity_id"])["entity_id"]
    if not ids.equals(S1["entity_id"].reset_index(drop=True)):
        print(f"{model}: different S1 records (other sample settings or data), not compared")
        continue
    r = pd.read_csv(f)
    runs[model] = r[r["method"] == "exact"]
    m = pd.read_csv(run_dir / "faiss" / "metric_f05.csv")
    ceilings[model] = m[m["method"].str.startswith("ceiling") & (m["buckets"] == "exact")].set_index("k")
    man = json.loads((run_dir / "manifest.json").read_text(encoding="utf-8"))
    timing += [{"model": model, "source": src, **t} for src, t in man.get("timing", {}).items()]
    # recall by script of the true candidate (India): from the saved exact buckets and the sample's links
    bk = pd.read_parquet(run_dir / "faiss" / "s1_buckets_exact.parquet", columns=["s1_entity_id", "rank", "candidate_entity_id"])
    cand = pd.concat([pd.read_parquet(run_dir / f"{x}_meta.parquet", columns=["entity_id", "true_s1", "tag_native_script", "country"])
                      for x in ("s2", "s3")])
    linked = cand[(cand["true_s1"] != "") & cand["true_s1"].isin(set(S1["entity_id"])) & (cand["country"] == "India")]
    rank = pd.Series(bk["rank"].to_numpy(), index=pd.MultiIndex.from_arrays([bk["s1_entity_id"], bk["candidate_entity_id"]]))
    got = rank.reindex(pd.MultiIndex.from_arrays([linked["true_s1"], linked["entity_id"]])).to_numpy()
    for k in (5, 10, 20, 50):
        for label, mask in (("Indian script", linked["tag_native_script"].to_numpy().astype(bool)),
                            ("Latin script", ~linked["tag_native_script"].to_numpy().astype(bool))):
            if mask.any():
                scripts.setdefault((k, label), {})[model] = float(np.mean(np.nan_to_num(got[mask], nan=1e9) < k))

if len(runs) < 2:
    print(f"only {list(runs)} has sample {SAMPLE_NAME}; set E5_MODELS to several models to compare")
else:
    COMPARE = pd.concat(runs, names=["model"]).reset_index(level=0).pivot_table(
        index="k", columns=["country", "model"], values="bucket recall (all pairs)")
    print(f"1. share of true matches inside the top-k buckets (exact search), sample {SAMPLE_NAME}")
    display(COMPARE.round(4))
    if scripts:
        SCRIPTS = pd.DataFrame(scripts).T
        SCRIPTS.index.names = ["k", "true candidate"]
        print("2. India: share of true matches inside the top-k buckets, by script of the candidate")
        display(SCRIPTS.round(4))
    keep = [c for c in next(iter(ceilings.values())).columns if c.startswith("macro F0.5")]
    CEILING = pd.concat({m: c[keep] for m, c in ceilings.items()}, axis=1)
    print("3. ceiling: best macro F0.5 a perfect matcher could reach on the top-k buckets")
    display(CEILING.round(4))
    TIMING = pd.DataFrame(timing)
    print("4. encoding: texts per second and seconds per source (loaded embeddings keep the time of their first run)")
    display(TIMING.pivot_table(index="source", columns="model", values=["texts_per_sec", "seconds"]))
    COMPARE.to_csv(EMB_ROOT / f"model_comparison_{SAMPLE_NAME}.csv")
    extra = {}
    for model in runs:
        f = EMB_ROOT / model / SAMPLE_NAME / "faiss" / "extra_recall.csv"
        if f.exists():
            extra[model] = pd.read_csv(f)
    if extra:
        EXTRA_COMPARE = pd.concat(extra, names=["model"]).reset_index(level=0)
        cols = ["pairs per S1", "true matches in bucket", "... Indian-script candidates", "... Latin-script candidates",
                "... empty-address candidates",
                "ceiling macro F0.5"]
        cols = [c for c in cols if c in EXTRA_COMPARE.columns]          # runs saved before a column existed
        order = list(dict.fromkeys(EXTRA_COMPARE["candidates"]))
        for country in sorted(EXTRA_COMPARE["country"].unique()):
            t = EXTRA_COMPARE[EXTRA_COMPARE["country"] == country].pivot_table(
                index="candidates", columns="model", values=cols).reindex(order)
            print(f"5. {country}: e5 top-k vs e5 top-30 + extra searches (the matcher judges 'e5 top-30 + all extra')")
            display(t[[c for c in cols if c in t.columns.get_level_values(0)]].round(4))
        EXTRA_COMPARE.to_csv(EMB_ROOT / f"model_comparison_extra_{SAMPLE_NAME}.csv", index=False)
    print("saved:", EMB_ROOT / f"model_comparison_{SAMPLE_NAME}.csv", "and", EMB_ROOT / f"model_comparison_extra_{SAMPLE_NAME}.csv")